### Import Dependencies

In [2]:
import openai
from qdrant_client import QdrantClient

### Embedding Function

In [3]:
def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

### Retrieval Function

In [4]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [5]:
def retrieve_data(query, collection_name='amazon-items-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

In [6]:
retrieve_context = retrieve_data(query='Which case do you recommend for Iphone 12?')

In [7]:
retrieve_context

{'retrieved_context_ids': ['B0B6R7CKVP',
  'B09PTX6461',
  'B0828S2KMV',
  'B09XSBZQVS',
  'B0BBVJJRHD'],
 'retrieved_context_scores': [0.4442664,
  0.43987697,
  0.4311992,
  0.41712195,
  0.41556954],
 'retrieved_context_texts': ['iCasso for New MacBook Air 13.6 inch Case 2022 A2681 M2 Chip with Retina Display，Glitter Hard Shell Case and Keyboard Skin Cover& Screen Protector -Pink [Compatibility]: Specifically designed for 2022 New Macbook Air 13.6 inch M2 Chip Model Number: A2681.Please kindly check the model number "A1xxx" on the back of the Macbook before purchasing this MacBook Air case. NOT Compatible with other models. [Protective Macbook Air M2 Case ]: Protects your MacBook Air 13 inch from accidental hard knocks and scratches. Fully access to all buttons and features. Plug your charger, cable or headset without removing the case. [Sleek design]: Embedded sparkling glitter crystalsprovide a radiant shimmer that won’t scratch off over time.smooth surface of macbook air m2 case 

### Format retrieved context function

In [15]:
def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

In [16]:
formatted_context = process_context(retrieve_context)

In [17]:
print(formatted_context)

- Product ID: B0B6R7CKVP, Product Rating: 4.1, Product Description: iCasso for New MacBook Air 13.6 inch Case 2022 A2681 M2 Chip with Retina Display，Glitter Hard Shell Case and Keyboard Skin Cover& Screen Protector -Pink [Compatibility]: Specifically designed for 2022 New Macbook Air 13.6 inch M2 Chip Model Number: A2681.Please kindly check the model number "A1xxx" on the back of the Macbook before purchasing this MacBook Air case. NOT Compatible with other models. [Protective Macbook Air M2 Case ]: Protects your MacBook Air 13 inch from accidental hard knocks and scratches. Fully access to all buttons and features. Plug your charger, cable or headset without removing the case. [Sleek design]: Embedded sparkling glitter crystalsprovide a radiant shimmer that won’t scratch off over time.smooth surface of macbook air m2 case 13.6 inch 2022 with endless shine. Simple sparkle looks rich and attractive. [Easy Installation]: Simple clip-on/off design, easy on and off without the added risk o

### Build the prompt template

In [18]:
def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

In [33]:
prompt = build_prompt(question='Do you have a case for Iphone 12?', formatted_context=formatted_context)

In [34]:
print(prompt)


    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    - Product ID: B0B6R7CKVP, Product Rating: 4.1, Product Description: iCasso for New MacBook Air 13.6 inch Case 2022 A2681 M2 Chip with Retina Display，Glitter Hard Shell Case and Keyboard Skin Cover& Screen Protector -Pink [Compatibility]: Specifically designed for 2022 New Macbook Air 13.6 inch M2 Chip Model Number: A2681.Please kindly check the model number "A1xxx" on the back of the Macbook before purchasing this MacBook Air case. NOT Compatible with other models. [Protective Macbook Air M2 Case ]: Protects your MacBook Air 13 inch from accidental hard knocks and scratches. Fully access to all buttons and features. Plug your charger, cable or hea

### Generate the answer function

In [35]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort='none'
    )

    return response.choices[0].message.content


In [36]:
print(generate_answer(prompt))

Yes. The available products include an MFi-certified Lightning iPhone charger cable that is compatible with iPhone 12 (and iPhone 12 Pro/12 Pro Max/12 Mini), but I don’t currently have an iPhone 12 case listed in the available products.


### Compile RAG pipeline

In [ ]:
def rag_pipeline(question, topk=5):
    retrieve_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieve_context)

    prompt = build_prompt(question=question, formatted_context=formatted_context)
    answer = generate_answer(prompt)

    return answer

In [41]:
print(rag_pipeline('Could you suggest me a drill for my home? Focus on products with rating above 4.5', 10))

I don’t have any drill/cordless drill products in the available items list. The only products above 4.5 that I can suggest instead are:

- B0CF57H28T (Rating 4.8): 1TB USB Flash Drive (metal, waterproof/rugged, USB 3.0)
- B0C996WY16 (Rating 4.7): Raymate Bluetooth Speaker (30W, IPX7 waterproof, ~1000 mins playtime)
- B0BRV544MV (Rating 4.4): not above 4.5, so I’m not including it

If you tell me what you need the drill for (wood, drywall, metal) and whether you want cordless vs corded, I can help you pick from the items that are available.
